# 4 · A paper figure  *(25 min)*

Panels, panel letters, export, a bit of polish, and finally: your `neuro` style, ready for every project.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.transforms import ScaledTranslation

import neurodata

plt.style.use(["default", "style_parts/1_basics.mplstyle", "style_parts/2_plots_color.mplstyle"])
mm = 1 / 25.4   # figsize is in inches: figsize=(120 * mm, 60 * mm)

## 1. 🎛 Several panels

Without a layout engine, labels of neighbouring panels collide. Switch to `"constrained"`:

In [ ]:
# 🎛 Change the value, then re-run the cell (Shift + Enter)
layout = "none"   # "none" or "constrained"

fig, axes = plt.subplots(2, 2, layout=layout)
for ax in axes.flat:
    ax.set(xlabel="Time (s)", ylabel="Rate (Hz)", title="Title")
plt.show()

🎛 `plt.subplot_mosaic` lets you **draw the layout as text**: one letter per panel, repeat a letter to make a
panel bigger, `;` starts a new row. Type your own layout, e.g. `AB;CC` or `AAB;AAC`.
Panel letters can't go in a style file, so `panel_label` is a small function you keep.

In [ ]:
def panel_label(ax, label):
    """Bold letter just outside the top-left corner of the panel (a bit larger than the other text)."""
    offset = ScaledTranslation(-18 / 72, 4 / 72, ax.figure.dpi_scale_trans)   # 18 pt left, 4 pt up
    ax.text(0, 1, label, transform=ax.transAxes + offset, fontsize="large", fontweight="bold", va="bottom")


# 🎛 Type your own layout, then re-run the cell (Shift + Enter)
mosaic = "AAB;CDB"

fig, axd = plt.subplot_mosaic(mosaic, figsize=(120 * mm, 60 * mm), layout="constrained")
for name, ax in axd.items():
    panel_label(ax, name)
plt.show()

## 2. 🎛 Two details you only notice when they're wrong

Timestamps from a long session get a hidden `+1.2e3` in the corner, and big counts get labels like
`200000`. Set them to `True`:

In [ ]:
t_lfp, x_lfp = neurodata.lfp()
window = (t_lfp > 3) & (t_lfp < 3.5)


# 🎛 Change the values, then re-run the cell (Shift + Enter)
real_numbers = False    # True or False
powers_of_ten = False   # True or False

rc = {"axes.formatter.useoffset": not real_numbers,
      "axes.formatter.use_mathtext": powers_of_ten, "axes.formatter.limits": (-3, 4) if powers_of_ten else (-5, 6)}
with mpl.rc_context(rc):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5, 1.8), layout="constrained", width_ratios=[2, 1])
    ax1.plot(1200 + t_lfp[window], x_lfp[window], linewidth=0.5)
    ax1.set(xlabel="Session time (s)", ylabel=r"LFP ($\mu$V)")
    ax2.bar(["M1", "M2", "M3"], [12_000, 95_000, 210_000], color="0.6")
    plt.show()

## 3. Export

Papers want **PDF** (vector: sharp at any zoom, editable in Illustrator/Inkscape); slides want **PNG**.
`pdf.fonttype: 42` keeps the text editable in the PDF, and `savefig.bbox: standard` keeps exactly the size
you asked for.

In [ ]:
%%writefile style_parts/3_paper.mplstyle
# ---- 3 · Paper ---------------------------------------------------------------
figure.constrained_layout.use: True

# fig.savefig("figure1") -> figure1.pdf, editable text, exact size
savefig.format: pdf
savefig.dpi: 300
savefig.bbox: standard
pdf.fonttype: 42

# Real numbers on axes (no "+1.2e3"), and x10^5 instead of 1e5
axes.formatter.useoffset: False
axes.formatter.use_mathtext: True
axes.formatter.limits: -3, 4

# Tighter spacing, made for 7 pt text
axes.labelpad: 2
xtick.major.pad: 2
ytick.major.pad: 2

## 4. Your style, ready for every project

Merge the three parts into **`my_styles/neuro.mplstyle`**, add a **talk** version that only changes sizes,
and copy both into matplotlib's style folder. From then on, `plt.style.use("neuro")` works in any notebook
on this computer.

In [ ]:
import os
import shutil

os.makedirs("my_styles", exist_ok=True)
parts = ["style_parts/1_basics.mplstyle", "style_parts/2_plots_color.mplstyle", "style_parts/3_paper.mplstyle"]
with open("my_styles/neuro.mplstyle", "w") as out:
    for part in parts:
        out.write(open(part).read() + "\n")

In [ ]:
%%writefile my_styles/neuro-talk.mplstyle
# Slides. Use on top of neuro: plt.style.use(["neuro", "neuro-talk"])
figure.figsize: 6.4, 4.0
font.size: 16
axes.labelsize: 16
axes.titlesize: 16
xtick.labelsize: 14
ytick.labelsize: 14
legend.fontsize: 14
axes.linewidth: 1.2
xtick.major.width: 1.2
ytick.major.width: 1.2
xtick.major.size: 6
ytick.major.size: 6
axes.labelpad: 4
lines.linewidth: 2.5
lines.markersize: 7
savefig.format: png

In [ ]:
stylelib = os.path.join(mpl.get_configdir(), "stylelib")   # matplotlib looks for styles here
os.makedirs(stylelib, exist_ok=True)
for name in ["neuro.mplstyle", "neuro-talk.mplstyle"]:
    shutil.copy(os.path.join("my_styles", name), stylelib)
plt.style.reload_library()
print([s for s in plt.style.available if s.startswith("neuro")])

## 5. 🎛 A small Figure 1, for a paper and for a talk

Three small habits finish it: `offset_spines` keeps data points off the axis lines, `color_legend` writes
each label in its line's color, and `panel_label` adds the letters. Switch the style: **the plot code stays
the same**.

In [ ]:
def offset_spines(ax):
    ax.spines[["left", "bottom"]].set_position(("outward", 3))   # 3 pt away from the data


def color_legend(ax):
    legend = ax.legend(handlelength=0, handletextpad=0)
    for text, line in zip(legend.get_texts(), legend.get_lines()):
        text.set_color(line.get_color())


trials = neurodata.spike_trains()
drug_trials = neurodata.spike_trains(peak_rate=30, sustained_rate=12, seed=1)
oris, responses = neurodata.orientation_tuning(n_neurons=3)
baseline, drug = neurodata.drug_experiment(n_animals=10)
edges = np.arange(0, 2.025, 0.025)

In [ ]:
# 🎛 Change the value, then re-run the cell (Shift + Enter)
style = "neuro"   # "neuro" or "neuro + neuro-talk"

with plt.style.context(["default"] + style.split(" + ")):
    size = (120 * mm, 90 * mm) if style == "neuro" else (8, 6)
    fig, axd = plt.subplot_mosaic("AB;CD", figsize=size)
    axd["A"].eventplot(trials, colors="black", linewidths=0.5)
    axd["A"].set(xlabel="Time (s)", ylabel="Trial")
    for label, tr in [("Control", trials), ("Drug", drug_trials)]:
        counts, _ = np.histogram(np.concatenate(tr), bins=edges)
        axd["B"].plot(edges[:-1], counts / (len(tr) * 0.025), label=label)
    axd["B"].set(xlabel="Time (s)", ylabel="Rate (Hz)")
    color_legend(axd["B"])
    axd["C"].plot(oris, responses.mean(axis=2).T, marker="o")
    axd["C"].set(xlabel="Orientation (°)", ylabel="Spikes / trial", xticks=[0, 90])
    axd["D"].boxplot([baseline, drug], widths=0.5)
    axd["D"].set_xticks([1, 2], ["Baseline", "Drug"])
    axd["D"].set(ylabel="Rate (Hz)", ylim=(0, None))
    for name, ax in axd.items():
        offset_spines(ax)
        panel_label(ax, name)
    fig.savefig("figure1")     # PDF for the paper, PNG for the talk: decided by the style
    plt.show()

## Take home

* In every new project: `plt.style.use("neuro")`. On a new computer: copy `my_styles/` and run the copy
  cell in section 4.
* Keep `panel_label`, `offset_spines` and `color_legend` in your analysis code.
* Before you submit a figure:
  - [ ] made at its **final size**, not shrunk later
  - [ ] every axis labelled **with units**; text 5–8 pt
  - [ ] color-blind-safe colors; same condition = same color everywhere; no `jet`
  - [ ] every data point shown when *n* is small
  - [ ] saved as **PDF**

More recipes (scale bars, LFP spectra, more colormaps, dark slides...): `05_toolbox.ipynb`.